In [2]:
import numpy as np

In [9]:
class Detector:
    def __init__(self, R, N, r, a):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        
        self.pmt_coords = self._generate_pmt_coordinates()

    def _generate_pmt_coordinates(self):
        #генерация с исользованием спирали Фиббоначи
        points = []
        phi = np.pi * (3. - np.sqrt(5.)) 
        
        for i in range(self.N):
            y = 1 - (i / float(self.N - 1)) * 2
            radius = np.sqrt(1 - y * y)
            theta = phi * i
            
            x = np.cos(theta) * radius
            z = np.sin(theta) * radius
            
            points.append([x * self.R, y * self.R, z * self.R])
            
        return np.array(points)

    def is_point_in_pmt(self, point):
        point = np.array(point)
        
        if not np.isclose(np.linalg.norm(point), self.R, rtol=1e-3):
            return False, None 
            
        distances = np.linalg.norm(self.pmt_coords - point, axis=1)
        
        min_dist = np.min(distances)
        closest_pmt_idx = np.argmin(distances)
        
        if min_dist <= self.r:
            return True, closest_pmt_idx
        else:
            return False, None
    def find_intersection(self, origin, directions):
        p = np.array(origin)
        # Векторизованное решение квадратного уравнения: |p + t*n|^2 = R^2
        # t^2 + 2(p*n)t + (p^2 - R^2) = 0
        b = np.dot(directions, p)
        c = np.dot(p, p) - self.R**2
        
        discriminant = b**2 - c
        #рассматриваем только положительный корень (отрицательный физически означает движение назад и нам не подходит)
        t = -b + np.sqrt(discriminant)
        
        # Координаты пересечения
        intersections = p + t[:, np.newaxis] * directions
        return intersections

    def generate_photoelectrons(self, num_hits_per_pmt):
        means = self.a * num_hits_per_pmt
        return np.random.poisson(means)

In [19]:
detector = Detector(R=2.0, N=40, r=0.1, a=0.5)
source = Source(Y=10000)

origin = (0.0, 0.0, 0.0)
E_event = 1.0 

directions = source.generate_photons(E_event)
print(f"Сгенерировано фотонов: {len(directions)}")

intersections = detector.find_intersection(origin, directions)

hits_per_pmt = get_hits_per_pmt(detector, intersections)
total_photons_hit = np.sum(hits_per_pmt)

print(f"Фотонов попало в ФЭУ: {total_photons_hit}")

pe_per_pmt = detector.generate_photoelectrons(hits_per_pmt)
total_Q = np.sum(pe_per_pmt)

print(f"Всего фотоэлектронов собрано (Q): {total_Q}")
Q_theory = 0.5 * 10000 * (40 * 0.1**2 / 2**2)



print(f"Смоделированное Q: {total_Q}")
print(f"Теоретическое Q : {Q_theory:.1f}")
#расхождение в 4 раза вызвано тем, что ФЭУ распределяются не по площади круга, как в формуле а по площади сверы, которая в 4 раза больше площади круга

Сгенерировано фотонов: 10000
Фотонов попало в ФЭУ: 248
Всего фотоэлектронов собрано (Q): 113
Смоделированное Q: 113
Теоретическое Q : 500.0


In [5]:
def get_hits_per_pmt(detector, intersections):
    hits = np.zeros(detector.N, dtype=int)
    dists = np.linalg.norm(intersections[:, np.newaxis, :] - detector.pmt_coords[np.newaxis, :, :], axis=2)
    closest_pmt_idx = np.argmin(dists, axis=1)
    min_dists = np.min(dists, axis=1)
    # Отбираем те фотоны, которые попали в радиус ФЭУ (r)
    valid_hits = min_dists <= detector.r
    hit_indices = closest_pmt_idx[valid_hits]
    hits = np.bincount(hit_indices, minlength=detector.N)
    
    return hits

In [7]:
class Source:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_photons(self, E):
        N_photons = int(self.Y * E)
        if N_photons <= 0:
            return np.empty((0, 3))
        vectors = np.random.normal(loc=0.0, scale=1.0, size=(N_photons, 3))
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        directions = vectors / norms
        
        return directions